#  Projekt 2 - Skrypt do przygotowania modelu głębokiego

Poniższy skrypt należy uruchomić w środowisku Google Colab.

## Model

*EfficientNet B0* model architecture from the *EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks* paper.

Parameters:

* weights (EfficientNet_B0_Weights, optional) – The pretrained weights to use. See EfficientNet_B0_Weights below for more details, and possible values. By default, no pre-trained weights are used.

* progress (bool, optional) – If True, displays a progress bar of the download to stderr. Default is True.

* **kwargs – parameters passed to the torchvision.models.efficientnet.EfficientNet base class. Please refer to the source code for more details about this class.

**EfficientNet_B0_Weights.IMAGENET1K_V1:**

These weights are ported from the original paper. Also available as EfficientNet_B0_Weights.DEFAULT.

acc@1: (on ImageNet-1K) 77.692
acc@5: (on ImageNet-1K) 93.532
categories: tench, goldfish, great white shark, … (997 omitted)
min_size: height=1, width=1
num_params: 5288548
GFLOPS: 0.39: 
File size 20.5 MB

In [ ]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

# Stworzenie modelu EfficientNet-B0 z wagami wytrenowanymi na ImageNet
model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)

**Transformacje** - są to przetworzenia tensora wejściowego. Możemy definiować własne transformacje, lub możemy skorzystać z transformacji dostarczonych przez autorów architektury.
Dla modelu *efficientnet_b0* z wagami *EfficientNet_B0_Weights.IMAGENET1K_V1* autorzy zastosowali:

* resize to resize_size=[256] z interpolation=InterpolationMode.BICUBIC, 
* central crop of crop_size=[224],
* normalizacja do [0.0, 1.0] i standaryzacja z mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225].


In [ ]:
# Pobranie transformacji dla modelu EfficientNet-B0 z wagami wytrenowanymi na ImageNet
transforms = EfficientNet_B0_Weights.IMAGENET1K_V1.transforms

Ponieważ chcemy wytrenować model na datasecie NMIST, a nie IMAGENET1K - stosujemy własne transformacje.

In [ ]:
from torchvision import transforms as t

# Mean oraz Std wyznaczono dla odwróconych obrazów MNIST (czarna cyfra na białym tle)
MEAN = (0.8693,)
STD = (0.3081,)

transforms = t.Compose([
    t.ToTensor(), # Konwersja obrazu do tensora z zakresem wartości [0, 1]
    t.functional.invert, # Odwrócenie kolorów obrazu (czerń->biel, biel->czerń, ...)
    t.Normalize(mean=MEAN, std=STD) # Standaryzacja obrazu
])

Przyjrzyjmy się budowie modelu:

In [ ]:
model

## Dataset

### Augmentacja danych

Przygotowane przez nas transformacje (*transforms*) pozwolą na przetworzenie obrazów i przekazanie ich do modelu głębokiego. Są to transformacje w pełni deterministyczne.

Jednakże w czasie trwania treningu warto stosować losowe transformacje, określane jako **augemntacje**. Poprzez losową modyfikację danych treningowych w czasie trwania treningu modelu jesteśmy w stanie poprawić zdolność uogólniania modelu, w efekcie uzyskując wyższą jakość wnioskowania i odporność na zmienność danych wejściowych.

In [ ]:
transforms # Istniejąca lista deterministycznych transformacji

transforms_with_augmentations = t.Compose([
    t.ToTensor(),  
    t.functional.invert,
    t.RandomRotation(degrees=15, fill=1), # Losowa rotacja obrazu w zakresie [-15, 15] stopni
    t.RandomAffine(degrees=15, translate=(0.20, 0.20), shear=10, fill=1), # Losowa rotacja obrazu w zakresie [-15, 15] stopni, przesunięcie obrazu w zakresie [-10%, 10%] w poziomie i pionie, oraz ścinanie obrazu w zakresie [-10, 10] stopni
    t.Normalize(mean=MEAN, std=STD),
])

### Stworzenie zestawów treningowego i walidacyjnego

In [ ]:
from torchvision.datasets import MNIST

dataset_train = MNIST(root='data', train=True, download=True, transform=transforms_with_augmentations)
dataset_val = MNIST(root='data', train=False, download=True, transform=transforms)

Przyjrzyjmy się obrazom:

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 3, figsize=(8, 9))

for idx,ax in enumerate(axes.ravel()):
    img, label = dataset_val[idx]
    img = img * STD[0] + MEAN[0]  # Odwrotność standardyzacji
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(f"Etykieta: {label}")
    ax.axis("off")

fig.suptitle("Przykładowe obrazy z zestawu walidacyjnego MNIST", fontsize=16)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(8, 9))

for idx,ax in enumerate(axes.ravel()):
    img, label = dataset_train[idx]
    img = img * STD[0] + MEAN[0]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(f"Etykieta: {label}")
    ax.axis("off")

fig.suptitle("Przykładowe obrazy z zestawu treningowego MNIST", fontsize=16)

plt.tight_layout()
plt.show()

Nasze dane treningowe to czarno-białe obrazy o wymiarze 28x28 przedstawiające ręcznie pisane cyfry.

## Definicja pętli treningowej i walidacyjnej

Aby przeprowadzić trening modelu musisymy zdefiniować hiperparametry i przygotować pętlę treningową.

In [ ]:
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

import torch.nn as nn
import torch.nn.functional as F

In [ ]:
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 10) # Zmiana liczby wyjść klasyfikatora na 10 (dla 10 klas MNIST)
model.features[0][0] = nn.Conv2d(1, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False) # Zmiana liczby kanałów wejściowych pierwszej warstwy konwolucyjnej na 1 (dla obrazów MNIST)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Wybór urządzenia (CPU lub GPU)

model.to(device) # Przeniesienie modelu na wybrane urządzenie

### Hiperparametry

* Optimizer - zarządza procesem uczenia maszynowego
* Criterion - kryterium (funkcja straty) użyte w czasie treningu
* Learning Rate (LR) - wartość mnożnika użyta podczas propagacji wstecznej
* Batch size - wielkość paczki wejściowej
* Num. epochs - liczba epok treningowych

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

batch_size = 256
num_epochs = 5
pin_memory = True if device.type == "cuda" else False

### Trening modelu

W tym przykładzie stosujemy bardzo prostą pętlę treningową. Stosujemy mechanizm zapis wag dla "najlepszego" modelu (tj. modelu, który uzyskuje najwyższą wartość *accuracy* na zestawie walidacyjnym).

In [ ]:
# Utworzenie DataLoaderów dla zbiorów treningowego i walidacyjnego
train_loader = DataLoader(dataset_train, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=pin_memory)
val_loader = DataLoader(dataset_val, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=pin_memory)

# Utworzenie zmiennych do logowania wyników treningu i ewaluacji
train_loss_history = []
val_loss_history = []
train_acc_history = []
val_acc_history = []

# Trening dla *num_epochs* epok
for epoch in range(1, num_epochs + 1):
    
    ### TRENING - aktualizacja wag modelu na podstawie zbioru treningowego

    model.train() # Zmiana trybu modelu na treningowy
    
    train_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch} | Train", leave=False) # Wizualizacja postępu treningu za pomocą paska postępu
    
    for x, y in pbar:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad() # Wyzerowanie gradientów dla wszystkich parametrów modelu przed obliczeniem nowych gradientów
        probs = model(x) # Wnioskowanie na modelu -> zwracanie prawdopodobieństw dla każdej klasy
        loss = criterion(probs, y) # Obliczanie straty (loss) na podstawie prawdopodobieństw i etykiet
        loss.backward() # Propagacja wsteczna - obliczanie gradientów dla wszystkich parametrów modelu
        optimizer.step() # Aktualizacja parametrów modelu na podstawie obliczonych gradientów

        train_loss += loss.item() * x.size(0) # Sumowanie straty dla wszystkich próbek w bieżącym batchu
        preds = probs.argmax(dim=1) # Predykcja etykiet na podstawie najwyższego prawdopodobieństwa dla każdej próbki (ARGMAX)
        correct += (preds == y).sum().item() # Zliczanie poprawnych predykcji w bieżącym batchu
        total += y.size(0) # Zliczanie wszystkich próbek w bieżącym batchu

        pbar.set_postfix(loss=loss.item(), acc=correct / total) # Wizualizacja aktualnego loss i accuracy w pasku postępu

    train_loss /= total
    train_acc = correct / total
    train_loss_history.append(train_loss)
    train_acc_history.append(train_acc)

    ### EWALUACJA - ocena modelu na zbiorze walidacyjnym (niezależnym od zbioru treningowego)
    
    model.eval() # Zmiana trybu modelu na ewaluacyjny (wyłączenie dropout i batch normalization)
    val_loss = 0.0
    correct_v = 0
    total_v = 0
    
    pbar = tqdm(val_loader, desc=f"Epoch {epoch} | Val", leave=False) # Wizualizacja postępu ewaluacji za pomocą paska postępu
    
    with torch.no_grad():
        for x, y in pbar:
            x, y = x.to(device), y.to(device)

            probs = model(x) # Wnioskowanie na modelu -> zwracanie prawdopodobieństw dla każdej klasy
            loss = criterion(probs, y) # Obliczanie straty (loss) na podstawie prawdopodobieństw i etykiet

            val_loss += loss.item() * x.size(0) # Sumowanie straty dla wszystkich próbek w bieżącym batchu
            preds = probs.argmax(dim=1) # Predykcja etykiet na podstawie najwyższego prawdopodobieństwa dla każdej próbki (ARGMAX)
            correct_v += (preds == y).sum().item() # Zliczanie poprawnych predykcji w bieżącym batchu
            total_v += y.size(0) # Zliczanie wszystkich próbek w bieżącym batchu
            
            pbar.set_postfix(loss=loss.item(), acc=correct_v / total_v) # Wizualizacja aktualnego loss i accuracy w pasku postępu

    val_loss /= total_v
    val_acc = correct_v / total_v
    val_loss_history.append(val_loss)
    val_acc_history.append(val_acc)

    if val_loss >= min(val_loss_history):
        torch.save(model.state_dict(), "mnist_efficientnet_b0.pth") # Zapisanie wag dla najlepszego modelu (najwyższa acc na zbiorze walidacyjnym)
    
    print(f"Epoch {epoch} | Train Loss: {train_loss:.4f} Train Acc.: {train_acc:.4f} Val. Loss: {val_loss:.4f} Val. Acc.: {val_acc:.4f}")

### Historia treningu

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, num_epochs + 1), train_loss_history, label="Train Loss")
axes[0].plot(range(1, num_epochs + 1), val_loss_history, label="Val. Loss")

axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(range(1, num_epochs + 1), train_acc_history, label="Train Acc.")
axes[1].plot(range(1, num_epochs + 1), val_acc_history, label="Val. Acc.")

axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

### Wczytanie wag

In [ ]:
def load_model(path, device):    
    model = efficientnet_b0(weights=None) # Stworzenie modelu EfficientNet-B0 bez wstępnie wytrenowanych wag
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 10) # Zmiana liczby wyjść klasyfikatora na 10 (dla 10 klas MNIST)
    model.features[0][0] = nn.Conv2d(1, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False) # Zmiana liczby kanałów wejściowych pierwszej warstwy konwolucyjnej na 1 (dla obrazów MNIST)

    model.load_state_dict(torch.load(path, map_location=device)) # Wczytanie wag modelu z pliku
    model.to(device) # Przeniesienie modelu na wybrane urządzenie (CPU lub GPU)
    model.eval() # Ustawienie modelu w tryb ewaluacji
    return model

model = load_model("mnist_efficientnet_b0.pth", device)


### Wizualizacja działania modelu

In [ ]:
def inference(model, image, device):
    model.eval()  # Ustawienie modelu w tryb ewaluacji
    with torch.no_grad():  # Wyłączenie obliczania gradientów
        image = image.to(device)  # Przeniesienie obrazu na odpowiednie urządzenie (CPU lub GPU)
        image = image.unsqueeze(0)  # Dodanie wymiaru batcha (1, C, H, W) dla pojedynczego obrazu
        probs = model(image)  # Wnioskowanie na modelu -> zwracanie prawdopodobieństw dla każdej klasy
        probs = torch.softmax(probs, dim=1) # Softmax - konwersja logitów na prawdopodobieństwa dla każdej klasy
        preds = probs.argmax(dim=1)  # Predykcja etykiet na podstawie najwyższego prawdopodobieństwa dla każdej próbki (ARGMAX)
    return preds
    
    # Uwaga - wynikiem wnioskowania jest tensor z indeksem klasy (0-9)!
    # W tym przypadku jest to tożsame z etykietą cyfry w zbiorze MNIST, np. 0, 1, 2, ..., 9.
    # W innych przypadkach może być konieczne mapowanie indeksu klasy na etykietę, np. za pomocą słownika lub listy.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(8, 9))

for idx,ax in enumerate(axes.ravel()):
    img, label = dataset_val[idx]

    preds = inference(model, img, device) 

    img = img * STD[0] + MEAN[0]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(f"Et.: {label} | Pred.: {preds.item()}")
    ax.axis("off")


fig.suptitle("Wyniki wnioskowania modelu", fontsize=16)

plt.tight_layout()
plt.show()

### Generacja tablicy testowej

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(5, 4, figsize=(8.27, 11.69))

for idx,ax in enumerate(axes.ravel()):
    img, label = dataset_val[idx]

    img = img * STD[0] + MEAN[0]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.axis("off")


fig.suptitle("MNIST Validation Set", fontsize=16)

plt.tight_layout()
plt.show()